# WP12 — Energy-Level Coefficient Falsifier

This notebook runs the repository audit for the WP11 one-sided high-tail target.
It does **not** prove L2–L3. The required coefficient is tautological and is used only as a falsification target.

$$b_{\rm req}^{(\theta)}=\frac{[N^{>K}-\theta\nu Y]_+}{X},\qquad Q_{\rm energy}=\frac{b_{\rm req}}{\sqrt G}.$$


In [ ]:
!git clone -q https://github.com/reggaesharkk/navier-stokes-bridge-audit.git
%cd navier-stokes-bridge-audit
!git checkout -q wp12-energy-coefficient-falsifier-20260925
!pip -q install numpy matplotlib pandas


In [ ]:
!python3 src/wp12_energy_coefficient_falsifier.py --output /content/wp12_energy_coefficient_results.json


In [ ]:
import json, pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
data=json.loads(Path('/content/wp12_energy_coefficient_results.json').read_text())
print('max scaling error =', data['max_scaling_error'])
rows=[]
for tr in data['traces']:
    rows.append({
        'scenario':tr['scenario'],'N':tr['cutoff'],'A':tr['amplitude_multiplier'],
        'theta':tr['theta'],'max_Q':tr['max_Q_energy'],
        'int_b':tr['integral_b_required'],'int_sqrtG':tr['integral_sqrt_G']})
df=pd.DataFrame(rows).sort_values('max_Q',ascending=False)
display(df.head(20))


In [ ]:
# Plot Q_energy(t) for the baseline amplitude A=1.
for theta in sorted(set(df.theta)):
    plt.figure(figsize=(8,5))
    for tr in data['traces']:
        if tr['amplitude_multiplier']==1.0 and tr['theta']==theta:
            label=f"{tr['scenario']} N={tr['cutoff']}"
            plt.plot([r['time'] for r in tr['rows']], [r['Q_energy'] for r in tr['rows']], label=label)
    plt.xlabel('time'); plt.ylabel('Q_energy'); plt.title(f'theta={theta}')
    plt.legend(); plt.grid(True, alpha=.25); plt.show()


In [ ]:
# Amplitude sweep: finite diagnostic only.
sel=df[(df.scenario=='combined_double_quarter_high') & (df.theta==0.5)]
for N in sorted(sel.N.unique()):
    x=sel[sel.N==N].sort_values('A')
    plt.plot(x.A,x.max_Q,marker='o',label=f'N={N}')
plt.xlabel('amplitude multiplier'); plt.ylabel('max Q_energy'); plt.xscale('log',base=2)
plt.title('Combined perturbed case, theta=0.5'); plt.legend(); plt.grid(True,alpha=.25); plt.show()


In [ ]:
# Exact image-sublattice scaling check.
scale_rows=[]
for d in data['dilation_checks']:
    scale_rows.append({
        'scenario':d['base'].get('scenario',''), 'N':d['N'],'theta':d['base']['reserve_fraction'],
        'max_error':d['maximum_error'],'Q_base':d['base']['Q_energy'],'Q_scaled':d['scaled']['Q_energy']})
display(pd.DataFrame(scale_rows))
print('This checks exact discrete dilation covariance only; it is not N→∞ convergence.')


## Interpretation rule

- A bounded-looking finite sweep does **not** prove a universal constant.
- A single explicit smooth family with unbounded $Q_{\rm energy}$ would refute the proposed $C\sqrt G$ coefficient.
- The open theorem remains a noncircular all-prefix bound on the **signed** high-advector transfer.
